In [5]:
from pyspark.sql import SparkSession
import json
import os
from pyspark.sql import functions as sf
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCESS_KEY = os.environ.get("AWS_ACCESS_KEY_ID", "forge-commerce-user")
SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "forge-commerce-pass")
S3_ENDPOINT = os.environ.get("AWS_S3_ENDPOINT", "http://minio:9000")
PREFIX = "customers"
RAW_BUCKET = "raw"
CLEANED_BUCKET = "cleaned"
CURATED_BUCKET = "curated"
RAW_PATH = f"s3a://{RAW_BUCKET}/{PREFIX}/"
CLEANED_PATH = f"s3a://{CLEANED_BUCKET}/{PREFIX}/"
CURATED_PATH = f"s3a://{CURATED_BUCKET}/{PREFIX}/"

In [6]:
spark = (
        SparkSession.builder.appName("test_payments")
        .master(os.environ.get("SPARK_MASTER", "spark://spark-master:7077"))
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        # Delta Lake configurations
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .getOrCreate()
    )

In [7]:
df_cleaned = spark.read.format("delta").load(CLEANED_PATH)
# df.printSchema()



# df.orderBy("customer_id").select("customer_id", "name", "created_at").show(20, truncate=False)

delta_table_exists = DeltaTable.isDeltaTable(spark, CURATED_PATH)

if not delta_table_exists:
    window_spec = Window.partitionBy("customer_id").orderBy(sf.col("created_at").asc())
    df_cleaned = (
        df_cleaned
        .withColumn("effective_from", sf.col("created_at"))
        .withColumn("effective_to", sf.lead("created_at").over(window_spec))
        .withColumn(
            "is_active",
            sf.when(sf.col("effective_to").isNull(), True).otherwise(False),
        )
    )

    df_cleaned.write.format("delta").partitionBy("creation_year", "creation_month").mode("overwrite").save(CURATED_PATH)
else:
    curated_delta_table = DeltaTable.forPath(spark, CURATED_PATH)
    df_curated = curated_delta_table.toDF()
    max_loaded = df_curated.agg({"created_at": "max"}).collect()[0][0]
    print("Last Loaded Date: ", max_loaded)
    df_cleaned_new = df_cleaned.filter(sf.col("created_at") > max_loaded)
    print("New Data: ", df_cleaned_new.count())
    customers_affected = df_cleaned_new.select("customer_id").distinct()
    print("Customers Affected: ", customers_affected.count())
    df_curated_active = (
        df_curated.filter(sf.col("is_active") == True).join(customers_affected, "customer_id", "inner")
    )
    df_union_cleaned_new_and_curated_active = df_curated_active.select(df_cleaned_new.columns).unionByName(df_cleaned_new)
    window_spec = Window.partitionBy("customer_id").orderBy(sf.col("created_at").asc())
    df_scd2 = (
        df_union_cleaned_new_and_curated_active
        .withColumn("effective_from", sf.col("created_at"))
        .withColumn("effective_to", sf.lead("created_at").over(window_spec))
        .withColumn(
            "is_active",
            sf.when(sf.col("effective_to").isNull(), True).otherwise(False),
        )
    )
    df_scd2.orderBy("customer_id", "created_at").select("customer_id", "name", "created_at", "effective_from", "effective_to", "is_active").show(25, truncate=False)
    # (
    #     curated_delta_table.alias("tgt")
    #     .merge(df_scd2.alias("src"), "tgt.customer_id = src.customer_id AND tgt.created_at = src.created_at")
    #     .whenMatchedUpdateAll()
    #     .whenNotMatchedInsertAll()
    #     .execute()
    # )


# df_curated.orderBy("customer_id").select("customer_id", "name", "created_at", "row_number", "effective_from", "effective_to", "is_active").show(25, truncate=False)
# df_cleaned.orderBy("customer_id", "created_at").select("customer_id", "name", "created_at").show(25, truncate=False)

Last Loaded Date:  2026-03-20 00:20:45


New Data:  2000


Customers Affected:  1000


+-----------+----------------+-------------------+-------------------+-------------------+---------+
|customer_id|name            |created_at         |effective_from     |effective_to       |is_active|
+-----------+----------------+-------------------+-------------------+-------------------+---------+
|1          |Brooke Bowman   |2026-03-20 00:20:44|2026-03-20 00:20:44|2026-03-23 22:23:12|false    |
|1          |Natalie Perkins |2026-03-23 22:23:12|2026-03-23 22:23:12|2026-03-23 22:30:05|false    |
|1          |Thomas Parsons  |2026-03-23 22:30:05|2026-03-23 22:30:05|NULL               |true     |
|2          |Stacey Ross     |2026-03-20 00:20:44|2026-03-20 00:20:44|2026-03-23 22:23:12|false    |
|2          |Sarah Castro    |2026-03-23 22:23:12|2026-03-23 22:23:12|2026-03-23 22:30:05|false    |
|2          |Jill Vasquez    |2026-03-23 22:30:05|2026-03-23 22:30:05|NULL               |true     |
|3          |Patrick Smith   |2026-03-20 00:20:44|2026-03-20 00:20:44|2026-03-23 22:23:12|f

In [4]:
spark.stop()